# Profile Cre Mdr Evaluability



Profile CRE and MDR evaluability

This file evaluates whether the ARMD source contains enough
organism and susceptibility information to apply the accepted
phenotypic CRE and MDR Pseudomonas aeruginosa rules.

No phenotype classifications are written to production models
at this stage.

1. Audit source organism labels eligible for CRE evaluation.

Purpose:
Identify every source label that would be assigned to a
CRE-eligible organism group before formalizing the mapping
in a version-controlled seed.

This CASE statement is temporary profiling logic. It will not
become the final production mapping without review.

In [1]:
%sql
with source_organisms as (

    select
        cohort_id,
        organism_source,

        count(*) as culture_organism_rows,
        count(distinct patient_key) as patients

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    group by
        cohort_id,
        organism_source

),

classified as (

    select
        cohort_id,
        organism_source,
        culture_organism_rows,
        patients,

        case

            -- Current and phenotype-qualified E. coli labels.
            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            -- Historical and current Klebsiella aerogenes labels.
            -- This condition must come before the broad
            -- Enterobacter and Klebsiella conditions.
            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            -- Other Klebsiella source labels.
            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            -- Enterobacter source labels other than historical
            -- Enterobacter aerogenes.
            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            -- Citrobacter source labels, including historical
            -- labels beginning with ZZZ.
            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            -- Only Serratia marcescens is included under the
            -- accepted CRE phenotype scope.
            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            -- Proteus species, including historical labels.
            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            -- Morganella scope is limited to M. morganii.
            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_organism_group

    from source_organisms

)

select
    cohort_id,
    cre_organism_group,
    organism_source,
    culture_organism_rows,
    patients

from classified

where cre_organism_group is not null

order by
    cohort_id,
    cre_organism_group,
    culture_organism_rows desc,
    organism_source;

,cohort_id,cre_organism_group,organism_source,culture_organism_rows,patients
0,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX,59,56
1,blood,Citrobacter spp.,CITROBACTER KOSERI,40,39
2,blood,Citrobacter spp.,CITROBACTER FREUNDII,19,18
3,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),9,9
4,blood,Citrobacter spp.,CITROBACTER KOSERI (CARBAPENEM RESISTANT),6,6
5,blood,Citrobacter spp.,CITROBACTER SPECIES,2,2
6,blood,Citrobacter spp.,CITROBACTER YOUNGAE,2,2
7,blood,Citrobacter spp.,ZZZCITROBACTER AMALONATICUS,2,2
8,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,254,240
9,blood,Enterobacter spp.,ENTEROBACTER CLOACAE,63,62


## Reviewed mapping coverage for CRE-eligible source labels

The preceding query uses broad source-label patterns to discover the
organism vocabulary that may be eligible for CRE evaluation.

Those broad patterns are appropriate for profiling, but the production
phenotype model should use an explicit, version-controlled mapping from
each source organism label to its reviewed base organism identity.

This check identifies CRE-eligible source labels that are not yet
included in the reviewed organism mapping.

For each CRE rule group, it also identifies:

- whether imipenem must be excluded from the phenotype rule;
- which unmapped labels already contain an explicit source-system
  carbapenem-resistant designation.

No production mapping is changed by this query.

In [1]:
%sql
with source_labels as (

    select distinct
        organism_source,
        mapping_status

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

classified as (

    select
        organism_source,
        mapping_status,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group

    from source_labels

),

unmapped as (

    select
        organism_source,
        cre_rule_group,

        case
            when organism_source like '%(CARBAPENEM RESISTANT)%'
                then 1
            else 0
        end as source_designated_cre

    from classified

    where cre_rule_group is not null
      and mapping_status = 'unmapped'

)

select
    cre_rule_group,

    case
        when cre_rule_group in (
            'Proteus spp.',
            'Morganella morganii'
        )
            then 1
        else 0
    end as exclude_imipenem_from_cre_rule,

    count(*) as unmapped_source_label_count,

    array_sort(
        collect_set(organism_source)
    ) as unmapped_source_labels,

    array_sort(
        collect_set(
            case
                when source_designated_cre = 1
                    then organism_source
            end
        )
    ) as source_designated_cre_labels

from unmapped

group by cre_rule_group

order by cre_rule_group

,cre_rule_group,exclude_imipenem_from_cre_rule,unmapped_source_label_count,unmapped_source_labels,source_designated_cre_labels
0,Citrobacter spp.,0,11,"[CITROBACTER AMALONATICUS COMPLEX, CITROBACTER BRAAKII, CITROBACTER FREUNDII, CITROBACTER FREUNDII COMPLEX, CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT), CITROBACTER KOSERI, CITROBACTER KOSERI (CARBAPENEM RESISTANT), CITROBACTER SPECIES, CITROBACTER YOUNGAE, ZZZCITROBACTER AMALONATICUS, ZZZCITROBACTER FARMERI]","[CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT), CITROBACTER KOSERI (CARBAPENEM RESISTANT)]"
1,Enterobacter spp.,0,7,"[ENTEROBACTER AMNIGENUS 1, ENTEROBACTER ASBURIAE, ENTEROBACTER CANCEROGENUS, ENTEROBACTER CLOACAE, ENTEROBACTER GERGOVIAE, ENTEROBACTER SPECIES, ENTEROBACTER TAYLORAE]",[]
2,Klebsiella aerogenes,0,3,"[KLEBSIELLA AEROGENES, KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT), ZZZENTEROBACTER AEROGENES]",[KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT)]
3,Klebsiella spp.,0,3,"[KLEBSIELLA OZAENAE, KLEBSIELLA PNEUMONIAE SSP. OZAENAE, KLEBSIELLA SPECIES]",[]
4,Morganella morganii,1,1,[MORGANELLA MORGANII],[]
5,Proteus spp.,1,4,"[PROTEUS SPECIES, PROTEUS VULGARIS GROUP, ZZZPROTEUS PENNERI, ZZZPROTEUS VULGARIS]",[]


## Reporting-group audit for Enterobacter and Citrobacter

The CRE phenotype definition and the antibiogram reporting groups serve
different purposes.

`cre_rule_group` identifies the broad organism group used to determine
whether a culture-organism record is eligible for CRE evaluation.

`organism_reporting_group` defines how reviewed organism identities are
grouped for patient-level deduplication and display in the antibiogram.

This query compares the exact ARMD source vocabulary with those reviewed
SOP reporting rules before any production mapping is changed.

In [2]:
%sql
with vocabulary as (

    select
        organism_source,

        max(organism_standardized)
            as current_standardized_name,

        max(source_qualifier)
            as current_source_qualifier,

        max(mapping_status)
            as current_mapping_status,

        count(*) as culture_organism_rows,

        count(distinct patient_key)
            as patients,

        array_sort(
            collect_set(cohort_id)
        ) as cohorts

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_source like 'ENTEROBACTER %'
       or organism_source like 'ZZZENTEROBACTER %'
       or organism_source like 'KLEBSIELLA AEROGENES%'
       or organism_source like 'CITROBACTER %'
       or organism_source like 'ZZZCITROBACTER %'

    group by organism_source

),

reviewed as (

    select
        *,

        case

            -- Current and historical Klebsiella aerogenes
            -- terminology. This remains separate from the
            -- Enterobacter cloacae complex.
            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'ENTEROBACTER AEROGENES%'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            -- Laboratory SOP reporting group:
            -- Enterobacter cloacae complex.
            when organism_source like 'ENTEROBACTER CLOACAE%'
              or organism_source like 'ENTEROBACTER ASBURIAE%'
              or organism_source like 'ENTEROBACTER HORMAECHEI%'
              or organism_source like 'ENTEROBACTER KOBEI%'
              or organism_source like 'ENTEROBACTER LUDWIGII%'
                then 'Enterobacter cloacae complex'

            -- Laboratory SOP reporting group:
            -- Citrobacter freundii complex.
            when organism_source like 'CITROBACTER BRAAKII%'
              or organism_source like 'CITROBACTER FARMERI%'
              or organism_source like 'ZZZCITROBACTER FARMERI%'
              or organism_source like 'CITROBACTER FREUNDII%'
              or organism_source like 'CITROBACTER KOSERI%'
              or organism_source like 'CITROBACTER YOUNGAE%'
                then 'Citrobacter freundii complex'

        end as proposed_sop_reporting_group,

        case
            when organism_source like '%(CARBAPENEM RESISTANT)%'
                then 1
            else 0
        end as source_designated_cre

    from vocabulary

)

select
    organism_source,
    current_standardized_name,
    current_source_qualifier,
    current_mapping_status,

    proposed_sop_reporting_group,
    source_designated_cre,

    culture_organism_rows,
    patients,
    cohorts

from reviewed

order by
    case
        when proposed_sop_reporting_group =
             'Enterobacter cloacae complex'
            then 1

        when proposed_sop_reporting_group =
             'Klebsiella aerogenes'
            then 2

        when proposed_sop_reporting_group =
             'Citrobacter freundii complex'
            then 3

        else 4
    end,

    organism_source

,organism_source,current_standardized_name,current_source_qualifier,current_mapping_status,proposed_sop_reporting_group,source_designated_cre,culture_organism_rows,patients,cohorts
0,ENTEROBACTER ASBURIAE,None,None,unmapped,Enterobacter cloacae complex,0,7,7,[urine]
1,ENTEROBACTER CLOACAE,None,None,unmapped,Enterobacter cloacae complex,0,380,346,"[blood, respiratory, urine]"
2,ENTEROBACTER CLOACAE COMPLEX,Enterobacter cloacae complex,None,reviewed,Enterobacter cloacae complex,0,1507,1285,"[blood, respiratory, urine]"
3,ENTEROBACTER CLOACAE COMPLEX (CARBAPENEM RESISTANT),Enterobacter cloacae complex,carbapenem_resistant,reviewed,Enterobacter cloacae complex,1,24,22,"[blood, respiratory, urine]"
4,KLEBSIELLA AEROGENES,None,None,unmapped,Klebsiella aerogenes,0,690,568,"[blood, respiratory, urine]"
5,KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT),None,None,unmapped,Klebsiella aerogenes,1,8,7,"[blood, respiratory, urine]"
6,ZZZENTEROBACTER AEROGENES,None,None,unmapped,Klebsiella aerogenes,0,725,639,"[blood, respiratory, urine]"
7,CITROBACTER BRAAKII,None,None,unmapped,Citrobacter freundii complex,0,31,30,"[respiratory, urine]"
8,CITROBACTER FREUNDII,None,None,unmapped,Citrobacter freundii complex,0,202,181,"[blood, respiratory, urine]"
9,CITROBACTER FREUNDII COMPLEX,None,None,unmapped,Citrobacter freundii complex,0,806,701,"[blood, respiratory, urine]"


## CRE source-label and AST concordance

Source organism labels containing `CARBAPENEM RESISTANT` are treated as
source-system phenotype designations, not as definitive classifications.

This analysis independently derives the expanded CRE phenotype from
resolved susceptibility endpoints and compares that result with the
source designation.

The independent AST classification is:

- `positive`: at least one qualifying carbapenem has a resolved
  `Resistant` interpretation;
- `not_detected`: at least one qualifying carbapenem has a resolved
  `Susceptible` or `Intermediate` interpretation, no qualifying
  carbapenem is Resistant, and no qualifying endpoint is missing,
  conflicting, Inconclusive, or Synergism;
- `not_evaluable`: no qualifying Susceptible, Intermediate, or Resistant
  endpoint is available, or a missing, conflicting, Inconclusive, or
  Synergism endpoint prevents a confident `not_detected` classification.

A positive classification takes precedence because one qualifying
Resistant result is sufficient under the accepted phenotype rule.

For Proteus species and Morganella morganii, imipenem and
imipenem/relebactam are not used in the CRE determination.

This audit is performed before patient-level deduplication because the
phenotype category must be assigned before selecting the first isolate
within each reporting category.

In [10]:
%sql
with organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group,

        case
            when upper(organism_source) like '%CARBAPENEM RESISTANT%'
                then 1
            else 0
        end as source_designated_cre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

cre_eligible as (

    select *
    from organism_scope
    where cre_rule_group is not null

),

qualifying_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,
        e.organism_source,
        e.cre_rule_group,
        e.source_designated_cre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from cre_eligible e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key

       and (

            (
                e.cre_rule_group in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam'
                )
            )

            or

            (
                e.cre_rule_group not in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Imipenem',
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam',
                    'Imipenem/Relebactam'
                )
            )

       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as qualifying_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as qualifying_carbapenem_results

    from qualifying_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_cre_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_cre = 1
             and ast_cre_status = 'positive'
                then 'concordant_positive'

            when source_designated_cre = 1
             and ast_cre_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_cre = 1
             and ast_cre_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_cre = 0
             and ast_cre_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_cre = 0
             and ast_cre_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as cre_concordance_status

    from ast_classified

)

select
    cohort_id,
    cre_rule_group,
    cre_concordance_status,

    count(*) as culture_organism_records,

    sum(source_designated_cre)
        as source_designated_cre_records,

    sum(
        case
            when ast_cre_status = 'positive' then 1
            else 0
        end
    ) as ast_cre_positive_records

from concordance

group by
    cohort_id,
    cre_rule_group,
    cre_concordance_status

order by
    cohort_id,
    cre_rule_group,
    cre_concordance_status

,cohort_id,cre_rule_group,cre_concordance_status,culture_organism_records,source_designated_cre_records,ast_cre_positive_records
0,blood,Citrobacter spp.,no_source_label_ast_not_detected,123,0,0
1,blood,Citrobacter spp.,no_source_label_not_evaluable,1,0,0
2,blood,Citrobacter spp.,source_positive_ast_not_detected,15,15,0
3,blood,Enterobacter spp.,ast_positive_without_source_label,8,0,8
4,blood,Enterobacter spp.,concordant_positive,3,3,3
5,blood,Enterobacter spp.,no_source_label_ast_not_detected,320,0,0
6,blood,Escherichia coli,ast_positive_without_source_label,1,0,1
7,blood,Escherichia coli,concordant_positive,4,4,4
8,blood,Escherichia coli,no_source_label_ast_not_detected,3334,0,0
9,blood,Escherichia coli,no_source_label_not_evaluable,1,0,0


## Detailed CRE discordance patterns

The aggregate concordance results show that source phenotype labels and
available categorical AST results do not always agree.

This section summarizes the exact qualifying carbapenem-result patterns
for records in one of three discordant states:

- source-designated CRE without a qualifying Resistant AST result;
- source-designated CRE that cannot be evaluated from available AST;
- AST-derived CRE without a resistant source label.

Results are grouped by organism source label and carbapenem pattern so
that the disagreement can be reviewed without publishing patient- or
culture-level identifiers.

These results will determine whether AST-derived status, source
designation, or an explicitly documented combination should control the
final phenotype field.

In [11]:
%sql
with organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group,

        case
            when upper(organism_source) like '%CARBAPENEM RESISTANT%'
                then 1
            else 0
        end as source_designated_cre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

cre_eligible as (

    select *
    from organism_scope
    where cre_rule_group is not null

),

qualifying_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,
        e.organism_source,
        e.cre_rule_group,
        e.source_designated_cre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from cre_eligible e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key

       and (

            (
                e.cre_rule_group in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam'
                )
            )

            or

            (
                e.cre_rule_group not in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Imipenem',
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam',
                    'Imipenem/Relebactam'
                )
            )

       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as qualifying_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as qualifying_carbapenem_results

    from qualifying_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_cre_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_cre = 1
             and ast_cre_status = 'positive'
                then 'concordant_positive'

            when source_designated_cre = 1
             and ast_cre_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_cre = 1
             and ast_cre_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_cre = 0
             and ast_cre_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_cre = 0
             and ast_cre_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as cre_concordance_status

    from ast_classified

)

select
    cohort_id,
    cre_rule_group,
    organism_source,

    ast_cre_status,
    cre_concordance_status,

    qualifying_endpoint_count,
    evaluable_endpoint_count,
    non_resistant_endpoint_count,
    non_evaluable_endpoint_count,
    resistant_endpoint_count,

    concat_ws(
        ', ',
        qualifying_carbapenem_results
    ) as qualifying_carbapenem_result_pattern,

    count(*) as culture_organism_records

from concordance

where cre_concordance_status in (
    'source_positive_ast_not_detected',
    'source_positive_not_evaluable',
    'ast_positive_without_source_label'
)

group by
    cohort_id,
    cre_rule_group,
    organism_source,
    ast_cre_status,
    cre_concordance_status,
    qualifying_endpoint_count,
    evaluable_endpoint_count,
    non_resistant_endpoint_count,
    non_evaluable_endpoint_count,
    resistant_endpoint_count,
    qualifying_carbapenem_results

order by
    cohort_id,
    culture_organism_records desc,
    cre_rule_group,
    organism_source,
    qualifying_carbapenem_result_pattern

,cohort_id,cre_rule_group,organism_source,ast_cre_status,cre_concordance_status,qualifying_endpoint_count,evaluable_endpoint_count,non_resistant_endpoint_count,non_evaluable_endpoint_count,resistant_endpoint_count,qualifying_carbapenem_result_pattern,culture_organism_records
0,blood,Citrobacter spp.,CITROBACTER KOSERI (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Imipenem=Susceptible, Meropenem=Susceptible",6
1,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,positive,ast_positive_without_source_label,3,3,2,0,1,"Ertapenem=Resistant, Imipenem=Susceptible, Meropenem=Susceptible",6
2,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Imipenem=Susceptible, Meropenem=Susceptible",5
3,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,3,3,3,0,0,"Ertapenem=Susceptible, Imipenem=Susceptible, Meropenem=Susceptible",2
4,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Ertapenem=Susceptible, Meropenem=Susceptible",2
5,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,positive,ast_positive_without_source_label,2,2,1,0,1,"Ertapenem=Resistant, Meropenem=Susceptible",2
6,blood,Escherichia coli,ESCHERICHIA COLI,positive,ast_positive_without_source_label,3,3,2,0,1,"Ertapenem=Resistant, Imipenem=Susceptible, Meropenem=Susceptible",1
7,blood,Klebsiella aerogenes,KLEBSIELLA AEROGENES,positive,ast_positive_without_source_label,2,2,1,0,1,"Ertapenem=Resistant, Meropenem=Susceptible",1
8,blood,Klebsiella spp.,KLEBSIELLA PNEUMONIAE,positive,ast_positive_without_source_label,3,3,1,0,2,"Ertapenem=Resistant, Meropenem/Vaborbactam=Susceptible, Meropenem=Resistant",1
9,blood,Klebsiella spp.,KLEBSIELLA PNEUMONIAE,positive,ast_positive_without_source_label,2,2,0,0,2,"Ertapenem=Resistant, Meropenem=Resistant",1


## S. aureus oxacillin-resistance endpoint coverage

Source organism labels containing `MRSA` or `OXACILLIN RESISTANT` are
treated as source-system phenotype designations rather than definitive
resistance classifications.

Before deriving an independent oxacillin-resistance phenotype, this
section profiles the availability and interpretation of the defining
oxacillin and cefoxitin endpoints.

The analysis identifies:

- which defining antibiotics are present in ARMD;
- susceptibility categories and reconciliation states by source label;
- culture-organism records with no defining endpoint;
- whether source-designated resistant records have usable AST evidence.

No production phenotype is assigned by this query.

In [1]:
%sql
with staphylococcus_aureus as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        case
            when upper(
                coalesce(source_qualifier, '')
            ) = 'MRSA'

              or upper(organism_source) like '%MRSA%'

              or upper(organism_source)
                    like '%OXACILLIN RESISTANT%'

                then 1

            else 0
        end as source_designated_oxacillin_resistant

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Staphylococcus aureus'

),

defining_results as (

    select
        s.culture_organism_key,
        s.culture_key,
        s.cohort_id,
        s.patient_key,

        s.organism_source,
        s.source_qualifier,
        s.source_designated_oxacillin_resistant,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from staphylococcus_aureus s

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on s.culture_organism_key = r.culture_organism_key

       and r.antibiotic in (
           'Oxacillin',
           'Cefoxitin'
       )

)

select
    cohort_id,
    organism_source,
    source_qualifier,
    source_designated_oxacillin_resistant,

    coalesce(
        antibiotic,
        '[no defining endpoint]'
    ) as defining_antibiotic,

    coalesce(
        resolution_status,
        'not_present'
    ) as resolution_status,

    coalesce(
        susceptibility,
        '[null]'
    ) as susceptibility,

    count(
        distinct culture_organism_key
    ) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from defining_results

group by
    cohort_id,
    organism_source,
    source_qualifier,
    source_designated_oxacillin_resistant,

    coalesce(
        antibiotic,
        '[no defining endpoint]'
    ),

    coalesce(
        resolution_status,
        'not_present'
    ),

    coalesce(
        susceptibility,
        '[null]'
    )

order by
    cohort_id,
    source_designated_oxacillin_resistant desc,
    organism_source,
    defining_antibiotic,
    resolution_status,
    susceptibility

,cohort_id,organism_source,source_qualifier,source_designated_oxacillin_resistant,defining_antibiotic,resolution_status,susceptibility,culture_organism_records,patients
0,blood,STAPH AUREUS {MRSA},MRSA,1,Oxacillin,conflict,[null],3,3
1,blood,STAPH AUREUS {MRSA},MRSA,1,Oxacillin,resolved,Resistant,418,366
2,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),small_colony_or_other_morphotype,0,Oxacillin,resolved,Resistant,3,3
3,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),small_colony_or_other_morphotype,0,Oxacillin,resolved,Susceptible,4,4
4,blood,STAPHYLOCOCCUS AUREUS,None,0,Oxacillin,conflict,[null],1,1
5,blood,STAPHYLOCOCCUS AUREUS,None,0,Oxacillin,missing,[null],1,1
6,blood,STAPHYLOCOCCUS AUREUS,None,0,Oxacillin,resolved,Resistant,200,187
7,blood,STAPHYLOCOCCUS AUREUS,None,0,Oxacillin,resolved,Susceptible,1297,1204
8,blood,STAPHYLOCOCCUS AUREUS,None,0,[no defining endpoint],not_present,[null],13,13
9,respiratory,STAPH AUREUS {MRSA},MRSA,1,Oxacillin,conflict,[null],17,13


## S. aureus oxacillin-resistance concordance

The endpoint-coverage analysis shows that source phenotype labels do not
fully determine oxacillin-resistance status.

Some source-designated MRSA records do not contain a resolved Resistant
defining endpoint, while some ordinary or morphotype-qualified
*Staphylococcus aureus* records contain resolved oxacillin resistance.

This section independently classifies oxacillin resistance using
reconciled oxacillin and cefoxitin endpoints and compares the result with
the source-system phenotype designation.

The independent AST classification is:

- `positive`: at least one defining endpoint is resolved as Resistant;
- `not_detected`: at least one defining endpoint is resolved as
  Susceptible or Intermediate, no endpoint is Resistant, and no defining
  endpoint is missing, conflicting, Inconclusive, or Synergism;
- `not_evaluable`: no usable defining endpoint is available, or unresolved
  evidence prevents a confident negative classification.

A positive classification takes precedence because one qualifying
Resistant interpretation is sufficient.

The source designation is retained for provenance and validation, but it
does not determine the final AST-derived phenotype.

In [2]:
%sql
with staphylococcus_aureus as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        case
            when upper(
                coalesce(source_qualifier, '')
            ) = 'MRSA'

              or upper(organism_source) like '%MRSA%'

              or upper(organism_source)
                    like '%OXACILLIN RESISTANT%'

                then 1

            else 0
        end as source_designated_oxacillin_resistant

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Staphylococcus aureus'

),

defining_results as (

    select
        s.culture_organism_key,
        s.culture_key,
        s.cohort_id,
        s.patient_key,

        s.organism_source,
        s.source_qualifier,
        s.source_designated_oxacillin_resistant,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from staphylococcus_aureus s

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on s.culture_organism_key = r.culture_organism_key

       and r.antibiotic in (
           'Oxacillin',
           'Cefoxitin'
       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,
        source_designated_oxacillin_resistant,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as defining_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as defining_ast_results

    from defining_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        source_qualifier,
        source_designated_oxacillin_resistant

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_oxacillin_resistance_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_oxacillin_resistant = 1
             and ast_oxacillin_resistance_status = 'positive'
                then 'concordant_positive'

            when source_designated_oxacillin_resistant = 1
             and ast_oxacillin_resistance_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_oxacillin_resistant = 1
             and ast_oxacillin_resistance_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_oxacillin_resistant = 0
             and ast_oxacillin_resistance_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_oxacillin_resistant = 0
             and ast_oxacillin_resistance_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as oxacillin_resistance_concordance_status

    from ast_classified

)

select
    cohort_id,
    organism_source,
    source_qualifier,

    source_designated_oxacillin_resistant,
    ast_oxacillin_resistance_status,
    oxacillin_resistance_concordance_status,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from concordance

group by
    cohort_id,
    organism_source,
    source_qualifier,
    source_designated_oxacillin_resistant,
    ast_oxacillin_resistance_status,
    oxacillin_resistance_concordance_status

order by
    cohort_id,
    source_designated_oxacillin_resistant desc,
    organism_source,
    oxacillin_resistance_concordance_status

,cohort_id,organism_source,source_qualifier,source_designated_oxacillin_resistant,ast_oxacillin_resistance_status,oxacillin_resistance_concordance_status,culture_organism_records,patients
0,blood,STAPH AUREUS {MRSA},MRSA,1,positive,concordant_positive,418,366
1,blood,STAPH AUREUS {MRSA},MRSA,1,not_evaluable,source_positive_not_evaluable,3,3
2,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),small_colony_or_other_morphotype,0,positive,ast_positive_without_source_label,3,3
3,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),small_colony_or_other_morphotype,0,not_detected,no_source_label_ast_not_detected,4,4
4,blood,STAPHYLOCOCCUS AUREUS,None,0,positive,ast_positive_without_source_label,200,187
5,blood,STAPHYLOCOCCUS AUREUS,None,0,not_detected,no_source_label_ast_not_detected,1297,1204
6,blood,STAPHYLOCOCCUS AUREUS,None,0,not_evaluable,no_source_label_not_evaluable,15,15
7,respiratory,STAPH AUREUS {MRSA},MRSA,1,positive,concordant_positive,890,379
8,respiratory,STAPH AUREUS {MRSA},MRSA,1,not_detected,source_positive_ast_not_detected,2,2
9,respiratory,STAPH AUREUS {MRSA},MRSA,1,not_evaluable,source_positive_not_evaluable,22,18


## Vancomycin-resistant Enterococcus endpoint coverage

Source organism labels containing `VANCO RESISTANT` or
`VANCOMYCIN RESISTANT` are treated as source-system phenotype
designations rather than definitive resistance classifications.

This analysis profiles vancomycin interpretations for the reviewed
Enterococcus species currently included in the reporting scope:

- *Enterococcus faecalis*
- *Enterococcus faecium*

The analysis identifies:

- whether a resolved vancomycin endpoint is available;
- the susceptibility and reconciliation status of that endpoint;
- whether source-designated VRE records have resistant AST evidence;
- whether records without a VRE source designation have resistant AST
  evidence;
- how often vancomycin is missing, conflicting, or absent.

Other Enterococcus species are not classified by this analysis. They
require separate terminology and eligibility review before inclusion.

No production phenotype is assigned by this query.

In [3]:
%sql
with enterococcus_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        organism_standardized,
        source_qualifier,

        case
            when upper(
                coalesce(source_qualifier, '')
            ) = 'VANCOMYCIN_RESISTANT'

              or upper(organism_source)
                    like '%VANCO RESISTANT%'

              or upper(organism_source)
                    like '%VANCOMYCIN RESISTANT%'

                then 1

            else 0
        end as source_designated_vre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized in (
        'Enterococcus faecalis',
        'Enterococcus faecium'
    )

),

vancomycin_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,

        e.organism_source,
        e.organism_standardized,
        e.source_qualifier,
        e.source_designated_vre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from enterococcus_scope e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key
       and r.antibiotic = 'Vancomycin'

)

select
    cohort_id,
    organism_standardized,
    organism_source,
    source_qualifier,
    source_designated_vre,

    coalesce(
        antibiotic,
        '[no vancomycin endpoint]'
    ) as defining_antibiotic,

    coalesce(
        resolution_status,
        'not_present'
    ) as resolution_status,

    coalesce(
        susceptibility,
        '[null]'
    ) as susceptibility,

    count(
        distinct culture_organism_key
    ) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from vancomycin_results

group by
    cohort_id,
    organism_standardized,
    organism_source,
    source_qualifier,
    source_designated_vre,

    coalesce(
        antibiotic,
        '[no vancomycin endpoint]'
    ),

    coalesce(
        resolution_status,
        'not_present'
    ),

    coalesce(
        susceptibility,
        '[null]'
    )

order by
    cohort_id,
    organism_standardized,
    source_designated_vre desc,
    organism_source,
    resolution_status,
    susceptibility

,cohort_id,organism_standardized,organism_source,source_qualifier,source_designated_vre,defining_antibiotic,resolution_status,susceptibility,culture_organism_records,patients
0,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS - VANCO RESISTANT,vancomycin_resistant,1,Vancomycin,resolved,Resistant,2,2
1,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,Vancomycin,resolved,Resistant,4,4
2,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,Vancomycin,resolved,Susceptible,562,524
3,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM - VANCO RESISTANT,vancomycin_resistant,1,Vancomycin,resolved,Inconclusive,1,1
4,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM - VANCO RESISTANT,vancomycin_resistant,1,Vancomycin,resolved,Resistant,67,62
5,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,Vancomycin,conflict,[null],1,1
6,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,Vancomycin,resolved,Resistant,85,77
7,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,Vancomycin,resolved,Susceptible,138,133
8,respiratory,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,Vancomycin,resolved,Susceptible,3,3
9,respiratory,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,Vancomycin,resolved,Resistant,1,1


## Vancomycin-resistant Enterococcus concordance

The endpoint-coverage analysis shows that source VRE designations are
usually supported by resistant vancomycin interpretations, but the source
labels do not identify every record with vancomycin resistance.

This section independently classifies vancomycin resistance using the
reconciled vancomycin endpoint and compares the result with the
source-system phenotype designation.

The independent AST classification is:

- `positive`: vancomycin is resolved as Resistant;
- `not_detected`: vancomycin is resolved as Susceptible or Intermediate;
- `not_evaluable`: vancomycin is absent, missing, conflicting,
  Inconclusive, Synergism, or otherwise unavailable for classification.

The source designation is retained for provenance and validation, but it
does not determine the final AST-derived phenotype.

The analysis remains limited to the reviewed reporting species
*Enterococcus faecalis* and *Enterococcus faecium*.

In [4]:
%sql
with enterococcus_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        organism_standardized,
        source_qualifier,

        case
            when upper(
                coalesce(source_qualifier, '')
            ) = 'VANCOMYCIN_RESISTANT'

              or upper(organism_source)
                    like '%VANCO RESISTANT%'

              or upper(organism_source)
                    like '%VANCOMYCIN RESISTANT%'

                then 1

            else 0
        end as source_designated_vre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized in (
        'Enterococcus faecalis',
        'Enterococcus faecium'
    )

),

vancomycin_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,

        e.organism_source,
        e.organism_standardized,
        e.source_qualifier,
        e.source_designated_vre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from enterococcus_scope e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key
       and r.antibiotic = 'Vancomycin'

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        organism_standardized,
        source_qualifier,
        source_designated_vre,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as defining_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when antibiotic is null
                    then 1

                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is null
                        then '[no vancomycin endpoint]'

                    else concat(
                        antibiotic,
                        '=',
                        coalesce(
                            susceptibility,
                            upper(resolution_status)
                        )
                    )
                end
            )
        ) as defining_ast_results

    from vancomycin_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        organism_standardized,
        source_qualifier,
        source_designated_vre

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_vancomycin_resistance_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_vre = 1
             and ast_vancomycin_resistance_status = 'positive'
                then 'concordant_positive'

            when source_designated_vre = 1
             and ast_vancomycin_resistance_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_vre = 1
             and ast_vancomycin_resistance_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_vre = 0
             and ast_vancomycin_resistance_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_vre = 0
             and ast_vancomycin_resistance_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as vre_concordance_status

    from ast_classified

)

select
    cohort_id,
    organism_standardized,
    organism_source,
    source_qualifier,

    source_designated_vre,
    ast_vancomycin_resistance_status,
    vre_concordance_status,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from concordance

group by
    cohort_id,
    organism_standardized,
    organism_source,
    source_qualifier,

    source_designated_vre,
    ast_vancomycin_resistance_status,
    vre_concordance_status

order by
    cohort_id,
    organism_standardized,
    source_designated_vre desc,
    organism_source,
    vre_concordance_status

,cohort_id,organism_standardized,organism_source,source_qualifier,source_designated_vre,ast_vancomycin_resistance_status,vre_concordance_status,culture_organism_records,patients
0,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS - VANCO RESISTANT,vancomycin_resistant,1,positive,concordant_positive,2,2
1,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,positive,ast_positive_without_source_label,4,4
2,blood,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,not_detected,no_source_label_ast_not_detected,562,524
3,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM - VANCO RESISTANT,vancomycin_resistant,1,positive,concordant_positive,67,62
4,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM - VANCO RESISTANT,vancomycin_resistant,1,not_evaluable,source_positive_not_evaluable,1,1
5,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,positive,ast_positive_without_source_label,85,77
6,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,not_detected,no_source_label_ast_not_detected,138,133
7,blood,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,not_evaluable,no_source_label_not_evaluable,1,1
8,respiratory,Enterococcus faecalis,ENTEROCOCCUS FAECALIS,None,0,not_detected,no_source_label_ast_not_detected,3,3
9,respiratory,Enterococcus faecium,ENTEROCOCCUS FAECIUM,None,0,positive,ast_positive_without_source_label,1,1


## MDR Pseudomonas aeruginosa category coverage

The MDR *Pseudomonas aeruginosa* phenotype is based on antimicrobial
categories rather than one defining antibiotic.

A category contributes positive evidence when at least one qualifying
antibiotic has a resolved `Intermediate` or `Resistant` interpretation.

The six evaluated categories are:

1. extended-spectrum cephalosporins;
2. fluoroquinolones;
3. aminoglycosides;
4. carbapenems;
5. piperacillin/tazobactam;
6. cefiderocol.

Gentamicin is not included in the accepted MDR rule.

This section profiles category-level evidence before assigning a final
MDR phenotype. Each culture-organism and category is classified as:

- `positive`: at least one qualifying endpoint is Intermediate or
  Resistant;
- `observed_non_positive`: at least one qualifying endpoint is
  Susceptible, with no Intermediate, Resistant, or unresolved endpoint;
- `not_evaluable`: a qualifying endpoint is missing, conflicting,
  Inconclusive, Synergism, or otherwise unusable, with no positive
  endpoint;
- `not_tested`: no qualifying antibiotic from the category is present.

`observed_non_positive` describes only the available AST evidence. It is
not yet treated as a definitive negative MDR category.

No production MDR phenotype is assigned by this query.

In [6]:
%sql
with category_definitions as (

    select *
    from values

        (
            1,
            'extended_spectrum_cephalosporins',
            'Cefepime'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftazidime'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftazidime/Avibactam'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftolozane/Tazobactam'
        ),

        (
            2,
            'fluoroquinolones',
            'Ciprofloxacin'
        ),
        (
            2,
            'fluoroquinolones',
            'Levofloxacin'
        ),

        (
            3,
            'aminoglycosides',
            'Tobramycin'
        ),

        (
            4,
            'carbapenems',
            'Imipenem'
        ),
        (
            4,
            'carbapenems',
            'Meropenem'
        ),
        (
            4,
            'carbapenems',
            'Imipenem/Relebactam'
        ),

        (
            5,
            'antipseudomonal_penicillin_combination',
            'Piperacillin/Tazobactam'
        ),

        (
            6,
            'siderophore_cephalosporin',
            'Cefiderocol'
        )

    as definitions(
        category_order,
        mdr_category,
        antibiotic
    )

),

categories as (

    select distinct
        category_order,
        mdr_category

    from category_definitions

),

pseudomonas_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Pseudomonas aeruginosa'

),

category_results as (

    select
        p.culture_organism_key,
        p.culture_key,
        p.cohort_id,
        p.patient_key,

        p.organism_source,
        p.source_qualifier,

        d.category_order,
        d.mdr_category,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from pseudomonas_scope p

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on p.culture_organism_key = r.culture_organism_key

    inner join category_definitions d
        on r.antibiotic = d.antibiotic

),

isolate_category_grid as (

    select
        p.culture_organism_key,
        p.culture_key,
        p.cohort_id,
        p.patient_key,

        p.organism_source,
        p.source_qualifier,

        c.category_order,
        c.mdr_category

    from pseudomonas_scope p

    cross join categories c

),

category_summary as (

    select
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,

        g.category_order,
        g.mdr_category,

        count(
            r.antibiotic
        ) as endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as positive_endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Susceptible'
                    then 1
                else 0
            end
        ) as susceptible_endpoint_count,

        sum(
            case
                when r.antibiotic is not null
                 and r.resolution_status in (
                     'missing',
                     'conflict'
                 )
                    then 1

                when r.antibiotic is not null
                 and r.resolution_status = 'resolved'
                 and (
                     r.susceptibility is null
                     or r.susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count

    from isolate_category_grid g

    left join category_results r
        on g.culture_organism_key = r.culture_organism_key
       and g.mdr_category = r.mdr_category

    group by
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,

        g.category_order,
        g.mdr_category

),

category_classified as (

    select
        *,

        case
            when positive_endpoint_count > 0
                then 'positive'

            when endpoint_count = 0
                then 'not_tested'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when susceptible_endpoint_count > 0
                then 'observed_non_positive'

            else 'not_evaluable'
        end as observed_category_status

    from category_summary

)

select
    cohort_id,
    organism_source,
    source_qualifier,

    category_order,
    mdr_category,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    sum(
        case
            when observed_category_status <> 'not_tested'
                then 1
            else 0
        end
    ) as category_tested_records,

    sum(
        case
            when observed_category_status = 'positive'
                then 1
            else 0
        end
    ) as category_positive_records,

    sum(
        case
            when observed_category_status =
                 'observed_non_positive'
                then 1
            else 0
        end
    ) as category_observed_non_positive_records,

    sum(
        case
            when observed_category_status = 'not_evaluable'
                then 1
            else 0
        end
    ) as category_not_evaluable_records,

    sum(
        case
            when observed_category_status = 'not_tested'
                then 1
            else 0
        end
    ) as category_not_tested_records,

    round(
        100.0
        * sum(
            case
                when observed_category_status <> 'not_tested'
                    then 1
                else 0
            end
        )
        / count(*),
        1
    ) as category_testing_coverage_pct

from category_classified

group by
    cohort_id,
    organism_source,
    source_qualifier,
    category_order,
    mdr_category

order by
    cohort_id,
    organism_source,
    category_order

,cohort_id,organism_source,source_qualifier,category_order,mdr_category,culture_organism_records,patients,category_tested_records,category_positive_records,category_observed_non_positive_records,category_not_evaluable_records,category_not_tested_records,category_testing_coverage_pct
0,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,1,extended_spectrum_cephalosporins,9,9,9,1,8,0,0,100.0
1,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,2,fluoroquinolones,9,9,9,4,5,0,0,100.0
2,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,3,aminoglycosides,9,9,9,0,9,0,0,100.0
3,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,4,carbapenems,9,9,9,4,5,0,0,100.0
4,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,5,antipseudomonal_penicillin_combination,9,9,5,1,4,0,4,55.6
5,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,6,siderophore_cephalosporin,9,9,0,0,0,0,9,0.0
6,blood,PSEUDOMONAS AERUGINOSA,None,1,extended_spectrum_cephalosporins,402,377,402,40,361,1,0,100.0
7,blood,PSEUDOMONAS AERUGINOSA,None,2,fluoroquinolones,402,377,402,70,330,2,0,100.0
8,blood,PSEUDOMONAS AERUGINOSA,None,3,aminoglycosides,402,377,363,5,358,0,39,90.3
9,blood,PSEUDOMONAS AERUGINOSA,None,4,carbapenems,402,377,402,47,355,0,0,100.0


## MDR Pseudomonas aeruginosa classification

The category-coverage analysis establishes the evidence available for
each of the six current NHSN MDR *Pseudomonas aeruginosa* categories.

This section combines those category-level results into one AST-derived
MDR status per source culture-organism record.

The classification is:

- `positive`: at least three categories contain Intermediate or Resistant
  evidence;
- `not_detected`: fewer than three categories are positive, and the record
  could not reach three positive categories even if every unknown category
  were positive;
- `not_evaluable`: fewer than three categories are currently positive, but
  missing, untested, conflicting, Inconclusive, or Synergism evidence
  leaves open the possibility that the record could meet the three-category
  threshold.

Unknown categories include both `not_tested` and `not_evaluable`
category states.

This evaluability rule is a conservative project policy layered onto the
current NHSN phenotype definition. It prevents incomplete testing from
being interpreted as definitive absence of MDR.

In [7]:
%sql
with category_definitions as (

    select *
    from values

        (
            1,
            'extended_spectrum_cephalosporins',
            'Cefepime'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftazidime'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftazidime/Avibactam'
        ),
        (
            1,
            'extended_spectrum_cephalosporins',
            'Ceftolozane/Tazobactam'
        ),

        (
            2,
            'fluoroquinolones',
            'Ciprofloxacin'
        ),
        (
            2,
            'fluoroquinolones',
            'Levofloxacin'
        ),

        (
            3,
            'aminoglycosides',
            'Tobramycin'
        ),

        (
            4,
            'carbapenems',
            'Imipenem'
        ),
        (
            4,
            'carbapenems',
            'Meropenem'
        ),
        (
            4,
            'carbapenems',
            'Imipenem/Relebactam'
        ),

        (
            5,
            'antipseudomonal_penicillin_combination',
            'Piperacillin/Tazobactam'
        ),

        (
            6,
            'siderophore_cephalosporin',
            'Cefiderocol'
        )

    as definitions(
        category_order,
        mdr_category,
        antibiotic
    )

),

categories as (

    select distinct
        category_order,
        mdr_category

    from category_definitions

),

pseudomonas_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Pseudomonas aeruginosa'

),

category_results as (

    select
        p.culture_organism_key,
        p.culture_key,
        p.cohort_id,
        p.patient_key,

        p.organism_source,
        p.source_qualifier,

        d.category_order,
        d.mdr_category,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from pseudomonas_scope p

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on p.culture_organism_key = r.culture_organism_key

    inner join category_definitions d
        on r.antibiotic = d.antibiotic

),

isolate_category_grid as (

    select
        p.culture_organism_key,
        p.culture_key,
        p.cohort_id,
        p.patient_key,

        p.organism_source,
        p.source_qualifier,

        c.category_order,
        c.mdr_category

    from pseudomonas_scope p

    cross join categories c

),

category_summary as (

    select
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,

        g.category_order,
        g.mdr_category,

        count(
            r.antibiotic
        ) as endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as positive_endpoint_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Susceptible'
                    then 1
                else 0
            end
        ) as susceptible_endpoint_count,

        sum(
            case
                when r.antibiotic is not null
                 and r.resolution_status in (
                     'missing',
                     'conflict'
                 )
                    then 1

                when r.antibiotic is not null
                 and r.resolution_status = 'resolved'
                 and (
                     r.susceptibility is null
                     or r.susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count

    from isolate_category_grid g

    left join category_results r
        on g.culture_organism_key = r.culture_organism_key
       and g.mdr_category = r.mdr_category

    group by
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,

        g.category_order,
        g.mdr_category

),

category_classified as (

    select
        *,

        case
            when positive_endpoint_count > 0
                then 'positive'

            when endpoint_count = 0
                then 'not_tested'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when susceptible_endpoint_count > 0
                then 'observed_non_positive'

            else 'not_evaluable'
        end as observed_category_status

    from category_summary

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        sum(
            case
                when observed_category_status = 'positive'
                    then 1
                else 0
            end
        ) as positive_category_count,

        sum(
            case
                when observed_category_status =
                     'observed_non_positive'
                    then 1
                else 0
            end
        ) as observed_non_positive_category_count,

        sum(
            case
                when observed_category_status = 'not_evaluable'
                    then 1
                else 0
            end
        ) as not_evaluable_category_count,

        sum(
            case
                when observed_category_status = 'not_tested'
                    then 1
                else 0
            end
        ) as not_tested_category_count,

        array_sort(
            collect_set(
                concat(
                    mdr_category,
                    '=',
                    observed_category_status
                )
            )
        ) as category_statuses

    from category_classified

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier

),

ast_classified as (

    select
        *,

        not_evaluable_category_count
        + not_tested_category_count
            as unknown_category_count,

        positive_category_count
        + not_evaluable_category_count
        + not_tested_category_count
            as maximum_possible_positive_category_count,

        case
            when positive_category_count >= 3
                then 'positive'

            when positive_category_count
               + not_evaluable_category_count
               + not_tested_category_count < 3
                then 'not_detected'

            else 'not_evaluable'
        end as ast_mdr_p_aeruginosa_status

    from isolate_summary

)

select
    cohort_id,
    organism_source,
    source_qualifier,

    ast_mdr_p_aeruginosa_status,

    positive_category_count,
    observed_non_positive_category_count,
    unknown_category_count,
    maximum_possible_positive_category_count,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from ast_classified

group by
    cohort_id,
    organism_source,
    source_qualifier,

    ast_mdr_p_aeruginosa_status,

    positive_category_count,
    observed_non_positive_category_count,
    unknown_category_count,
    maximum_possible_positive_category_count

order by
    cohort_id,
    organism_source,
    ast_mdr_p_aeruginosa_status,
    positive_category_count desc,
    unknown_category_count

,cohort_id,organism_source,source_qualifier,ast_mdr_p_aeruginosa_status,positive_category_count,observed_non_positive_category_count,unknown_category_count,maximum_possible_positive_category_count,culture_organism_records,patients
0,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_detected,1,4,1,2,2,2
1,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_detected,0,5,1,1,1,1
2,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_detected,0,4,2,2,1,1
3,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_evaluable,2,3,1,3,1,1
4,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,not_evaluable,1,3,2,3,3,3
5,blood,MUCOID PSEUDOMONAS AERUGINOSA,mucoid,positive,3,2,1,4,1,1
6,blood,PSEUDOMONAS AERUGINOSA,None,not_detected,1,4,1,2,55,48
7,blood,PSEUDOMONAS AERUGINOSA,None,not_detected,0,5,1,1,262,252
8,blood,PSEUDOMONAS AERUGINOSA,None,not_detected,0,4,2,2,33,32
9,blood,PSEUDOMONAS AERUGINOSA,None,not_evaluable,2,3,1,3,14,14


## Source-system MDR designation audit

The AST-derived MDR phenotype has been calculated independently from the
six reviewed antimicrobial categories.

This audit searches the source terminology for explicit MDR or
multidrug-resistant designations before a concordance comparison is
attempted.

A zero-row result means that source-system MDR status is not available
from the organism terminology in this ARMD release. In that case, the
project will retain only the independently derived AST phenotype.

In [8]:
%sql
select
    organism_source,
    source_qualifier,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    array_sort(
        collect_set(cohort_id)
    ) as cohorts

from workspace.micro_dev.int_mapped_cohort_culture_organism

where organism_standardized = 'Pseudomonas aeruginosa'

  and (
         upper(organism_source) like '%MDR%'

      or upper(organism_source)
             like '%MULTI%DRUG%'

      or upper(organism_source)
             like '%MULTIDRUG%'

      or upper(
             coalesce(source_qualifier, '')
         ) like '%MDR%'

      or upper(
             coalesce(source_qualifier, '')
         ) like '%MULTI%DRUG%'

      or upper(
             coalesce(source_qualifier, '')
         ) like '%MULTIDRUG%'
  )

group by
    organism_source,
    source_qualifier

order by
    culture_organism_records desc,
    organism_source

,organism_source,source_qualifier,culture_organism_records,patients,cohorts


## Source-system ESBL designation audit

This audit determines whether ARMD preserves that source-system ESBL
designation in the organism label or source qualifier.

A zero-row result means that confirmed source ESBL status is not
available in this ARMD release. In that case, an AST-derived
extended-spectrum-cephalosporin-resistant phenotype may still be
possible, but it must not be presented as a confirmed ESBL mechanism.

In [9]:
%sql
select
    organism_source,
    organism_standardized,
    source_qualifier,
    mapping_status,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    array_sort(
        collect_set(cohort_id)
    ) as cohorts

from workspace.micro_dev.int_mapped_cohort_culture_organism

where
       upper(organism_source) like '%ESBL%'

    or upper(organism_source)
           like '%EXTENDED%SPECTRUM%BETA%LACTAMASE%'

    or upper(organism_source)
           like '%EXTENDED%SPECTRUM%B-LACTAMASE%'

    or upper(
           coalesce(source_qualifier, '')
       ) like '%ESBL%'

    or upper(
           coalesce(source_qualifier, '')
       ) like '%EXTENDED%SPECTRUM%BETA%LACTAMASE%'

group by
    organism_source,
    organism_standardized,
    source_qualifier,
    mapping_status

order by
    culture_organism_records desc,
    organism_source

,organism_source,organism_standardized,source_qualifier,mapping_status,culture_organism_records,patients,cohorts


## Extended-spectrum cephalosporin-resistant Enterobacterales coverage

The current NHSN extended-spectrum cephalosporin-resistant
Enterobacterales phenotype applies to:

- *Escherichia coli*;
- *Klebsiella aerogenes*;
- *Klebsiella oxytoca*;
- *Klebsiella pneumoniae*;
- *Enterobacter* species.

The phenotype is based on a resolved `Resistant` interpretation to at
least one of the following:

- cefepime;
- ceftriaxone;
- cefotaxime;
- ceftazidime;
- ceftazidime/avibactam;
- ceftolozane/tazobactam.

This section profiles endpoint availability before assigning an
isolate-level phenotype.

Each organism-antibiotic endpoint is classified as:

- `resistant`: the endpoint is resolved as Resistant;
- `observed_non_resistant`: the endpoint is resolved as Susceptible or
  Intermediate;
- `not_evaluable`: the endpoint is missing, conflicting, Inconclusive,
  Synergism, or otherwise unusable;
- `not_tested`: no endpoint is present.

CRE and extended-spectrum cephalosporin resistance are initially derived
as independent phenotypes. A separate reporting hierarchy will later
give CRE precedence when assigning mutually exclusive antibiogram
display categories.

In [10]:
%sql
with qualifying_antibiotics as (

    select *
    from values

        (1, 'Cefepime'),
        (2, 'Ceftriaxone'),
        (3, 'Cefotaxime'),
        (4, 'Ceftazidime'),
        (5, 'Ceftazidime/Avibactam'),
        (6, 'Ceftolozane/Tazobactam')

    as qualifying(
        antibiotic_order,
        antibiotic
    )

),

organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'ENTEROBACTER AEROGENES%'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA OXYTOCA%'
                then 'Klebsiella oxytoca'

            when organism_source like 'KLEBSIELLA PNEUMONIAE%'
                then 'Klebsiella pneumoniae'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

        end as esce_organism_group

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

eligible_organisms as (

    select *
    from organism_scope

    where esce_organism_group is not null

),

organism_antibiotic_grid as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,

        e.organism_source,
        e.source_qualifier,
        e.esce_organism_group,

        q.antibiotic_order,
        q.antibiotic

    from eligible_organisms e

    cross join qualifying_antibiotics q

),

endpoint_classified as (

    select
        g.culture_organism_key,
        g.culture_key,
        g.cohort_id,
        g.patient_key,

        g.organism_source,
        g.source_qualifier,
        g.esce_organism_group,

        g.antibiotic_order,
        g.antibiotic,

        r.resolution_status,
        r.susceptibility,

        case
            when r.antibiotic is null
                then 'not_tested'

            when r.resolution_status = 'resolved'
             and r.susceptibility = 'Resistant'
                then 'resistant'

            when r.resolution_status = 'resolved'
             and r.susceptibility in (
                 'Susceptible',
                 'Intermediate'
             )
                then 'observed_non_resistant'

            else 'not_evaluable'
        end as observed_endpoint_status

    from organism_antibiotic_grid g

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on g.culture_organism_key = r.culture_organism_key
       and g.antibiotic = r.antibiotic

)

select
    cohort_id,
    esce_organism_group,
    organism_source,
    source_qualifier,

    antibiotic_order,
    antibiotic,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    sum(
        case
            when observed_endpoint_status <> 'not_tested'
                then 1
            else 0
        end
    ) as tested_endpoint_records,

    sum(
        case
            when observed_endpoint_status in (
                'resistant',
                'observed_non_resistant'
            )
                then 1
            else 0
        end
    ) as evaluable_endpoint_records,

    sum(
        case
            when observed_endpoint_status = 'resistant'
                then 1
            else 0
        end
    ) as resistant_records,

    sum(
        case
            when observed_endpoint_status =
                 'observed_non_resistant'
                then 1
            else 0
        end
    ) as observed_non_resistant_records,

    sum(
        case
            when observed_endpoint_status = 'not_evaluable'
                then 1
            else 0
        end
    ) as not_evaluable_records,

    sum(
        case
            when observed_endpoint_status = 'not_tested'
                then 1
            else 0
        end
    ) as not_tested_records,

    round(
        100.0
        * sum(
            case
                when observed_endpoint_status <> 'not_tested'
                    then 1
                else 0
            end
        )
        / count(*),
        1
    ) as testing_coverage_pct,

    round(
        100.0
        * sum(
            case
                when observed_endpoint_status in (
                    'resistant',
                    'observed_non_resistant'
                )
                    then 1
                else 0
            end
        )
        / count(*),
        1
    ) as evaluable_coverage_pct

from endpoint_classified

group by
    cohort_id,
    esce_organism_group,
    organism_source,
    source_qualifier,

    antibiotic_order,
    antibiotic

order by
    cohort_id,
    esce_organism_group,
    organism_source,
    antibiotic_order

,cohort_id,esce_organism_group,organism_source,source_qualifier,antibiotic_order,antibiotic,culture_organism_records,patients,tested_endpoint_records,evaluable_endpoint_records,resistant_records,observed_non_resistant_records,not_evaluable_records,not_tested_records,testing_coverage_pct,evaluable_coverage_pct
0,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,1,Cefepime,1,1,1,1,0,1,0,0,100.0,100.0
1,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,2,Ceftriaxone,1,1,1,1,0,1,0,0,100.0,100.0
2,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,3,Cefotaxime,1,1,1,1,0,1,0,0,100.0,100.0
3,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,4,Ceftazidime,1,1,0,0,0,0,0,1,0.0,0.0
4,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,5,Ceftazidime/Avibactam,1,1,0,0,0,0,0,1,0.0,0.0
5,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,6,Ceftolozane/Tazobactam,1,1,0,0,0,0,0,1,0.0,0.0
6,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,1,Cefepime,1,1,1,1,0,1,0,0,100.0,100.0
7,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,2,Ceftriaxone,1,1,1,1,0,1,0,0,100.0,100.0
8,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,3,Cefotaxime,1,1,0,0,0,0,0,1,0.0,0.0
9,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,4,Ceftazidime,1,1,1,1,0,1,0,0,100.0,100.0


## Extended-spectrum cephalosporin-resistant Enterobacterales classification

The endpoint-coverage analysis shows that qualifying extended-spectrum
cephalosporins are represented unevenly across organisms and time.
Traditional agents such as ceftriaxone, cefepime, cefotaxime, and
ceftazidime provide most of the available evidence, while newer
combination agents are tested less frequently.

The NHSN `ESCEall_AR` phenotype is positive when an eligible
Enterobacterales isolate has a resolved `Resistant` interpretation to at
least one qualifying extended-spectrum cephalosporin.

This project classifies each source culture-organism record as:

- `positive`: at least one qualifying endpoint is resolved as Resistant;
- `not_detected`: at least one qualifying endpoint is resolved as
  Susceptible or Intermediate, no endpoint is Resistant, and no tested
  qualifying endpoint is unresolved;
- `not_evaluable`: no usable qualifying endpoint is available, or a
  missing, conflicting, Inconclusive, Synergism, or otherwise unusable
  tested endpoint prevents a confident `not_detected` classification.

Untested qualifying drugs do not by themselves make a record
non-evaluable. The classification reflects the qualifying AST evidence
actually present in the ARMD extract.

CRE and ESCE-R are derived independently. A separate reporting hierarchy
will later give CRE precedence when mutually exclusive antibiogram
display categories are assigned.

In [11]:
%sql
with organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'ENTEROBACTER AEROGENES%'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA OXYTOCA%'
                then 'Klebsiella oxytoca'

            when organism_source like 'KLEBSIELLA PNEUMONIAE%'
                then 'Klebsiella pneumoniae'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

        end as esce_organism_group

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

eligible_organisms as (

    select *
    from organism_scope

    where esce_organism_group is not null

),

qualifying_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,

        e.organism_source,
        e.source_qualifier,
        e.esce_organism_group,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from eligible_organisms e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key

       and r.antibiotic in (
           'Cefepime',
           'Ceftriaxone',
           'Cefotaxime',
           'Ceftazidime',
           'Ceftazidime/Avibactam',
           'Ceftolozane/Tazobactam'
       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,
        esce_organism_group,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as qualifying_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when antibiotic is not null
                 and resolution_status in (
                     'missing',
                     'conflict'
                 )
                    then 1

                when antibiotic is not null
                 and resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as qualifying_esce_results

    from qualifying_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,

        organism_source,
        source_qualifier,
        esce_organism_group

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_esce_enterobacterales_status

    from isolate_summary

)

select
    cohort_id,
    esce_organism_group,
    organism_source,
    source_qualifier,

    ast_esce_enterobacterales_status,

    count(*) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients,

    round(
        100.0
        * count(*)
        / sum(count(*)) over (
            partition by
                cohort_id,
                esce_organism_group
        ),
        1
    ) as pct_within_cohort_organism_group

from ast_classified

group by
    cohort_id,
    esce_organism_group,
    organism_source,
    source_qualifier,
    ast_esce_enterobacterales_status

order by
    cohort_id,
    esce_organism_group,
    organism_source,
    ast_esce_enterobacterales_status

,cohort_id,esce_organism_group,organism_source,source_qualifier,ast_esce_enterobacterales_status,culture_organism_records,patients,pct_within_cohort_organism_group
0,blood,Enterobacter spp.,ENTEROBACTER AMNIGENUS 1,None,not_detected,1,1,0.3
1,blood,Enterobacter spp.,ENTEROBACTER CANCEROGENUS,None,not_detected,1,1,0.3
2,blood,Enterobacter spp.,ENTEROBACTER CLOACAE,None,not_detected,55,54,16.6
3,blood,Enterobacter spp.,ENTEROBACTER CLOACAE,None,positive,8,8,2.4
4,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,None,not_detected,200,189,60.4
5,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,None,not_evaluable,17,17,5.1
6,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,None,positive,37,36,11.2
7,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX (CARBAPENEM RESISTANT),carbapenem_resistant,positive,3,3,0.9
8,blood,Enterobacter spp.,ENTEROBACTER GERGOVIAE,None,not_detected,4,4,1.2
9,blood,Enterobacter spp.,ENTEROBACTER SPECIES,None,not_detected,5,5,1.5
